# Stock Return Prediction - Price-Based Baseline

**Objective:** Build a clean baseline model that predicts next-day stock returns using only historical closing prices.

## Problem Definition

- **Input:** Past W days of daily closing prices
- **Output:** Next-day expected return
- **Return formula:** `return_{t+1} = (P_{t+1} - P_t) / P_t`
- **Task type:** Regression (continuous output)

## Pipeline Overview

1. Download historical price data (Yahoo Finance)
2. Create sliding windows from price history
3. Split data chronologically (train/val/test)
4. Train LSTM model with MSE loss
5. Evaluate with MSE, MAE, and Directional Accuracy

**Note:** This baseline uses ONLY prices. No news, no graphs, no federated learning yet!

## 1. Environment Setup and Dependencies

In [ ]:
# Import required libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Add src to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

# Import our custom modules
from src.data.price_loader import PriceDataLoader, normalize_prices
from src.data.splits import split_by_time
from src.models.price_model import LSTMPriceModel
from src.evaluation.metrics import compute_all_metrics, print_metrics, evaluate_model
from src.utils.seed import set_seed

print("✓ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Set random seed for reproducibility
set_seed(42)

## 2. Data Loading from Yahoo Finance

We'll download Apple (AAPL) stock data from 2015 to 2023.

In [ ]:
# Configuration
TICKER = "AAPL"
START_DATE = "2015-01-01"
END_DATE = "2023-12-31"
WINDOW_SIZE = 20  # Past 20 days

# Create data loader
loader = PriceDataLoader(
    ticker=TICKER,
    start_date=START_DATE,
    end_date=END_DATE,
    window_size=WINDOW_SIZE
)

# Download and prepare data
X, y, dates = loader.load_and_prepare()

## 3. Visualize Price History and Returns Distribution

In [ ]:
# Visualize data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price history (sample window)
axes[0, 0].plot(X[100, :, 0], marker='o', linewidth=2)
axes[0, 0].set_title('Sample Price Window (20 days)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Day')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].grid(True, alpha=0.3)

# All prices over time
sample_prices = X[:, -1, 0]  # Last price from each window
axes[0, 1].plot(dates, sample_prices, linewidth=1.5, color='blue')
axes[0, 1].set_title(f'{TICKER} Price History', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].grid(True, alpha=0.3)

# Returns distribution
axes[1, 0].hist(y, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Returns Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Return')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero return')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Returns over time
axes[1, 1].plot(dates, y, linewidth=0.5, alpha=0.6)
axes[1, 1].set_title('Returns Over Time', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Return')
axes[1, 1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nReturn Statistics:")
print(f"Mean: {y.mean():.6f}")
print(f"Std: {y.std():.6f}")
print(f"Min: {y.min():.6f}")
print(f"Max: {y.max():.6f}")

## 4. Train/Validation/Test Split (Time-Based)

Split data chronologically: 70% train, 15% validation, 15% test.

In [ ]:
# Time-based split
splits = split_by_time(
    X, y, dates,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
)

X_train, y_train, dates_train = splits['train']
X_val, y_val, dates_val = splits['val']
X_test, y_test, dates_test = splits['test']

## 5. Normalize Data

Normalize using training set statistics only to prevent information leakage.

In [ ]:
# Normalize using training set statistics
X_train_norm, X_val_norm, X_test_norm, (mean, std) = normalize_prices(
    X_train, X_val, X_test
)

## 6. Create PyTorch DataLoaders

In [ ]:
# Create datasets
BATCH_SIZE = 32

train_dataset = TensorDataset(
    torch.FloatTensor(X_train_norm),
    torch.FloatTensor(y_train)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val_norm),
    torch.FloatTensor(y_val)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_norm),
    torch.FloatTensor(y_test)
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 7. Define LSTM Model

Simple LSTM that takes a price window and outputs a predicted return.

In [ ]:
# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LSTMPriceModel(
    input_dim=1,
    hidden_dim=64,
    num_layers=2,
    dropout=0.2
)
model.to(device)

print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {model.count_parameters():,}")

## 8. Training Configuration

In [ ]:
# Training hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 0.001

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Training configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Loss function: MSE")
print(f"  Optimizer: Adam")

## 9. Training Loop

Train the model and track training/validation losses.

In [ ]:
# Training loop
train_losses = []
val_losses = []
best_val_loss = float('inf')

print("Starting training...")
print("="*60)

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)
            
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()

print("="*60)
print(f"Training completed!")
print(f"Best validation loss: {best_val_loss:.6f}")

# Load best model
model.load_state_dict(best_model_state)

## 10. Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Evaluate on Test Set

Generate predictions and compute all metrics: MSE, MAE, RMSE, and Directional Accuracy.

In [ ]:
# Generate predictions
predictions, targets = evaluate_model(model, test_loader, device)

# Compute metrics
test_metrics = compute_all_metrics(predictions, targets)
print_metrics(test_metrics, "Test Set")

## 12. Visualize Predictions vs True Returns

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Scatter plot: Predicted vs True
axes[0, 0].scatter(targets, predictions, alpha=0.5, s=10)
axes[0, 0].plot([-0.1, 0.1], [-0.1, 0.1], 'r--', linewidth=2, label='Perfect prediction')
axes[0, 0].set_xlabel('True Return', fontsize=12)
axes[0, 0].set_ylabel('Predicted Return', fontsize=12)
axes[0, 0].set_title('Predicted vs True Returns', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Time series: First 100 samples
axes[0, 1].plot(targets[:100], label='True', linewidth=2)
axes[0, 1].plot(predictions[:100], label='Predicted', linewidth=2, alpha=0.7)
axes[0, 1].set_xlabel('Sample', fontsize=12)
axes[0, 1].set_ylabel('Return', fontsize=12)
axes[0, 1].set_title('Sample Predictions (First 100)', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Prediction errors
errors = predictions - targets
axes[1, 0].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Prediction Error', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Prediction Error Distribution', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Directional accuracy over time
correct_direction = np.sign(predictions) == np.sign(targets)
window = 50
rolling_accuracy = np.convolve(correct_direction, np.ones(window)/window, mode='valid')
axes[1, 1].plot(rolling_accuracy, linewidth=2)
axes[1, 1].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Random (50%)')
axes[1, 1].set_xlabel('Sample', fontsize=12)
axes[1, 1].set_ylabel('Accuracy', fontsize=12)
axes[1, 1].set_title(f'Rolling Directional Accuracy (Window={window})', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Summary and Next Steps

### What We Accomplished

✅ **Clean data pipeline**
- Downloaded historical price data from Yahoo Finance
- Created sliding windows for time series prediction
- Split data chronologically (no information leakage)

✅ **Baseline LSTM model**
- Simple architecture: LSTM + FC layer
- Predicts next-day returns (regression)
- Trained with MSE loss and Adam optimizer

✅ **Proper evaluation**
- MSE, MAE, RMSE for regression performance
- Directional accuracy for trading relevance
- Visualizations to understand model behavior

### Key Insights

1. **Returns are noisy:** Stock returns are inherently difficult to predict from prices alone
2. **Directional accuracy matters:** In trading, getting the sign right is more important than exact values
3. **Baseline established:** This provides a clean starting point for extensions

### Next Steps

This baseline will be extended with:

1. **News Data Integration**
   - Scrape financial news articles
   - Extract sentiment or embeddings
   - Combine with price data (multimodal)

2. **Graph Transformer**
   - Model relationships between stocks
   - Use graph neural networks
   - Capture market structure

3. **Federated Learning**
   - Distribute training across multiple clients
   - Privacy-preserving aggregation
   - Scalable to large datasets

4. **Advanced Features**
   - Technical indicators (RSI, MACD, etc.)
   - Volume and volatility features
   - Cross-asset features

### Conclusion

**We now have a working, evaluated baseline model!** 🎉

The model can:
- Take past 20 days of prices
- Predict next-day return
- Achieve reasonable performance

This clean baseline makes it easy to add complexity incrementally and measure improvements.